In [5]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from typing import Optional
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import WebBaseLoader
import json

load_dotenv()
llm = ChatGroq(model="llama-3.3-70b-versatile")

In [6]:
class UserInfo(BaseModel):
    name: Optional[str] = Field(default=None, description="Name of the person")
    age: Optional[int] = Field(default=None, description="Age of the person, calculate from birth year if needed using 2026")

In [7]:
parser = PydanticOutputParser(pydantic_object=UserInfo)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract info from context only.\n{format_instructions}\n\nContext:\n{context}"),
    ("human", "{question}")
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from langchain_chroma import Chroma

class Embeddings():
    def __init__(self):
        self._ef = DefaultEmbeddingFunction()
    def embed_documents(self, texts):
        return [[float(x) for x in v] for v in self._ef(texts)]
    def embed_query(self, text):
        return [float(x) for x in self._ef([text])[0]]

docs = WebBaseLoader("https://en.wikipedia.org/wiki/Elon_Musk").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vector_store = Chroma.from_documents(chunks, Embeddings(), collection_name="elon")
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

def ask_structured(question):
    retrieved = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in retrieved)
    return chain.invoke({"context": context, "question": question})

res = ask_structured("what is Elon Musk's name and age?")
print(json.dumps(res.model_dump(), indent=2))

{
  "name": "Elon Musk",
  "age": 54
}


In [9]:
docs = WebBaseLoader("https://en.wikipedia.org/wiki/Elon_Musk").load()
context = docs[0].page_content[:4000]

res = chain.invoke({"context": context, "question": "what is Elon Musk's name and age?"})
print(json.dumps(res.model_dump(), indent=2))

{
  "name": "Elon Musk",
  "age": 54
}
